In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-basis-constructor
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pengenalan fractional gates

*Estimasi penggunaan: di bawah 30 detik pada prosesor Heron r2 (CATATAN: Ini hanya estimasi. Waktu aktual bisa berbeda.)*
## Latar Belakang
### Fractional Gates pada IBM QPU
Fractional gates adalah quantum gates berparameter yang memungkinkan eksekusi langsung rotasi sudut sembarang (dalam batas tertentu),
menghilangkan kebutuhan untuk menguraikannya menjadi beberapa basis gates.
Dengan memanfaatkan interaksi alami antara qubit fisik, pengguna dapat mengimplementasikan unitary tertentu secara lebih efisien pada perangkat keras.

IBM Quantum&reg; Heron QPU mendukung fractional gates berikut:
- $R_{ZZ}(\theta)$ untuk $0 < \theta < \pi / 2$
- $R_X(\theta)$ untuk nilai real $\theta$ apa pun

Gates ini dapat secara signifikan mengurangi kedalaman maupun durasi quantum circuit.
Keunggulannya terutama terasa pada aplikasi yang banyak mengandalkan $R_{ZZ}$ dan $R_X$,
seperti simulasi Hamiltonian, Quantum Approximate Optimization Algorithm (QAOA), dan metode quantum kernel.
Dalam tutorial ini, kita fokus pada quantum kernel sebagai contoh praktis.

### Keterbatasan
Fractional gates saat ini merupakan fitur eksperimental dan memiliki beberapa batasan:
- $R_{ZZ}$ dibatasi pada sudut dalam rentang $0 < \theta < \pi / 2$.
- Penggunaan fractional gates tidak didukung untuk [dynamic circuits](/guides/classical-feedforward-and-control-flow), [Pauli twirling](/guides/error-mitigation-and-suppression-techniques#pauli-twirling), [probabilistic error cancellation](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) (PEC), dan [zero-noise extrapolation](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) (ZNE) (menggunakan [probabilistic error amplification](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) (PEA)).

Fractional gates memerlukan alur kerja yang berbeda dibandingkan pendekatan standar.
Tutorial ini menjelaskan cara bekerja dengan fractional gates melalui aplikasi praktis.

Lihat referensi berikut untuk detail lebih lanjut tentang fractional gates.
- [Fractional gates](/guides/fractional-gates)
- [Kapan *tidak* menggunakan fractional gates](/guides/fractional-gates#when-not-to-use)

## Ikhtisar
Alur kerja untuk menggunakan fractional gates umumnya mengikuti alur kerja [Qiskit patterns](/guides/intro-to-patterns).
Perbedaan utamanya adalah semua sudut RZZ harus memenuhi batasan $0 < \theta \leq \pi/2$.
Ada dua pendekatan untuk memastikan kondisi ini terpenuhi.
Tutorial ini berfokus pada dan merekomendasikan pendekatan kedua.

### 1. Buat nilai parameter yang memenuhi batasan sudut RZZ
Jika kamu yakin semua sudut RZZ berada dalam rentang yang valid, kamu bisa mengikuti alur kerja Qiskit patterns standar.
Dalam hal ini, kamu cukup mengirimkan nilai parameter sebagai bagian dari PUB. Alur kerjanya adalah sebagai berikut.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import unitary_overlap
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

Jika kamu mencoba mengirimkan PUB yang menyertakan gate RZZ dengan sudut di luar rentang yang valid, kamu akan menemui pesan error seperti:
```
'The instruction rzz is supported only for angles in the range [0, pi/2], but an angle (20.0) outside of this range has been requested; via parameter value(s) γ[0]=10.0, substituted in parameter expression 2.0*γ[0].'
```
Untuk menghindari error ini, pertimbangkan pendekatan kedua yang dijelaskan di bawah.

### 2. Tetapkan nilai parameter ke circuit sebelum transpilasi
Paket `qiskit-ibm-runtime` menyediakan transpiler pass khusus bernama [`FoldRzzAngle`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/transpiler-passes-fold-rzz-angle).
Pass ini mengubah quantum circuit agar semua sudut RZZ memenuhi batasan sudut RZZ.
Jika kamu menyediakan backend ke `generate_preset_pass_manager` atau `transpile`, Qiskit secara otomatis menerapkan `FoldRzzAngle` ke quantum circuit.
Ini mengharuskan kamu menetapkan nilai parameter ke quantum circuit sebelum transpilasi.
Alur kerjanya adalah sebagai berikut.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)  # backend should be a heron device or later
backend_name = backend.name
backend_c = service.backend(backend_name)  # w/o fractional gates
backend_f = service.backend(
    backend_name, use_fractional_gates=True
)  # w/ fractional gates
print(f"Backend: {backend_name}")
print(f"No fractional gates: {backend_c.basis_gates}")
print(f"With fractional gates: {backend_f.basis_gates}")
if "rzz" not in backend_f.basis_gates:
    print(f"Backend {backend_name} does not support fractional gates")

Backend: ibm_fez
No fractional gates: ['cz', 'id', 'rz', 'sx', 'x']
With fractional gates: ['cz', 'id', 'rx', 'rz', 'rzz', 'sx', 'x']


Perlu diperhatikan bahwa alur kerja ini menimbulkan biaya komputasi lebih tinggi dibandingkan pendekatan pertama, karena melibatkan penetapan nilai parameter ke quantum circuit dan penyimpanan circuit yang sudah terikat parameter secara lokal.
Selain itu, ada isu yang diketahui dalam Qiskit di mana transformasi gate RZZ bisa gagal dalam skenario tertentu. Untuk solusinya, silakan lihat bagian [Pemecahan Masalah](#troubleshooting).
Tutorial ini menunjukkan cara menggunakan fractional gates melalui pendekatan kedua dengan contoh yang terinspirasi dari metode quantum kernel.
Untuk lebih memahami di mana quantum kernel kemungkinan berguna, kami merekomendasikan membaca [Liu, Arunachalam & Temme (2021).](https://www.nature.com/articles/s41567-021-01287-z)

Kamu juga bisa mempelajari tutorial [Quantum kernel training](/tutorials/quantum-kernel-training) dan pelajaran [Quantum kernels](/learning/courses/quantum-machine-learning/quantum-kernel-methods) dalam kursus Quantum machine learning di IBM Quantum Learning.

### Persyaratan
Sebelum memulai tutorial ini, pastikan kamu sudah menginstal hal-hal berikut:
- Qiskit SDK v2.0 atau lebih baru, dengan dukungan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.37 atau lebih baru (`pip install qiskit-ibm-runtime`)
- Qiskit Basis Constructor (`pip install qiskit_basis_constructor`)

### Setup

In [3]:
optimization_level = 2
shots = 2000
reps = 3
rng = np.random.default_rng(seed=123)

In [4]:
def my_zz_feature_map(num_qubits: int, reps: int = 1) -> QuantumCircuit:
    x = ParameterVector("x", num_qubits * reps)
    qc = QuantumCircuit(num_qubits)
    qc.h(range(num_qubits))
    for k in range(reps):
        K = k * num_qubits
        for i in range(num_qubits):
            qc.rz(x[i + K], i)
        pairs = [(i, i + 1) for i in range(num_qubits - 1)]
        for i, j in pairs[0::2] + pairs[1::2]:
            qc.rzz((np.pi - x[i + K]) * (np.pi - x[j + K]), i, j)
    return qc


def quantum_kernel(num_qubits: int, reps: int = 1) -> QuantumCircuit:
    qc = my_zz_feature_map(num_qubits, reps=reps)
    inner_product = unitary_overlap(qc, qc, "x", "y", insert_barrier=True)
    inner_product.measure_all()
    return inner_product


def random_parameters(inner_product: QuantumCircuit) -> np.ndarray:
    return np.tile(rng.random(inner_product.num_parameters // 2), 2)


def fidelity(result) -> float:
    ba = result.data.meas
    return ba.get_int_counts().get(0, 0) / ba.num_shots

Quantum kernel circuits and their corresponding parameter values are generated for systems with 4 to 40 qubits, and their fidelities are subsequently evaluated.

In [5]:
qubits = list(range(4, 44, 4))
circuits = [quantum_kernel(i, reps=reps) for i in qubits]
params = [random_parameters(circ) for circ in circuits]

## Workflow dengan fractional gates
### Langkah 1: Petakan input klasik ke masalah kuantum
#### Circuit quantum kernel
Di bagian ini, kita mengeksplorasi circuit quantum kernel menggunakan Gate RZZ untuk memperkenalkan workflow fractional gates.

Kita mulai dengan membangun Circuit kuantum untuk menghitung entri-entri individual dari matriks kernel.
Ini dilakukan dengan menggabungkan Circuit ZZ feature map dengan unitary overlap.
Fungsi kernel mengambil vektor dalam ruang yang dipetakan fitur dan mengembalikan hasil kali dalamnya sebagai entri matriks kernel:
$$K(x, y) = \langle \Phi(x) | \Phi(y) \rangle,$$
di mana $|\Phi(x)\rangle$ merepresentasikan state kuantum yang dipetakan fitur.

Kita membangun Circuit ZZ feature map secara manual menggunakan Gate RZZ.
Meskipun Qiskit menyediakan `zz_feature_map` bawaan, fitur ini belum mendukung Gate RZZ per Qiskit v2.0.2 ([lihat issue](https://github.com/Qiskit/qiskit/issues/14469)).

Selanjutnya, kita menghitung fungsi kernel untuk input yang identik — misalnya, $K(x, x) = 1$.
Pada komputer kuantum yang bising, nilai ini bisa kurang dari 1 akibat noise.
Hasil yang lebih mendekati 1 menunjukkan noise yang lebih rendah dalam eksekusi.
Dalam tutorial ini, kita menyebut nilai ini sebagai *fidelitas*, yang didefinisikan sebagai
$$\text{fidelity} = K(x, x).$$

In [6]:
circuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/b3d6341a-0.avif" alt="Output of the previous code cell" />

In the standard Qiskit patterns workflow, parameter values are typically passed to the Sampler or Estimator primitive as part of a PUB.
However, when using a backend that supports fractional gates, these parameter values must be explicitly assigned to the quantum circuit prior to transpilation.

In [7]:
b_qc = [
    circ.assign_parameters(param) for circ, param in zip(circuits, params)
]
b_qc[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/6c9c1977-0.avif" alt="Output of the previous code cell" />

Circuit quantum kernel beserta nilai parameter-nya dibuat untuk sistem dengan 4 hingga 40 Qubit, lalu fidelitasnya dievaluasi.

In [8]:
backend_f = service.backend(name=backend_name, use_fractional_gates=True)
# pm_f includes `FoldRzzAngle` pass
pm_f = generate_preset_pass_manager(
    optimization_level=optimization_level, backend=backend_f
)

In [9]:
t_qc_f = pm_f.run(b_qc)
print(t_qc_f[0].count_ops())
t_qc_f[0].draw("mpl", fold=-1)

OrderedDict([('rz', 35), ('rzz', 18), ('x', 13), ('rx', 9), ('measure', 4), ('barrier', 2)])


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/a18e5c70-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/fractional-gates/extracted-outputs/b3d6341a-0.avif)

Dalam workflow Qiskit patterns standar, nilai parameter biasanya diteruskan ke primitif Sampler atau Estimator sebagai bagian dari PUB.
Namun, ketika menggunakan Backend yang mendukung fractional gates, nilai parameter tersebut harus ditetapkan secara eksplisit ke Circuit kuantum sebelum transpilasi.

In [10]:
nnl_f = [qc.num_nonlocal_gates() for qc in t_qc_f]
depth_f = [qc.depth() for qc in t_qc_f]
duration_f = [
    qc.estimate_duration(backend_f.target, unit="u") for qc in t_qc_f
]

![Output of the previous code cell](../docs/images/tutorials/fractional-gates/extracted-outputs/6c9c1977-0.avif)

### Langkah 2: Optimalkan masalah untuk eksekusi perangkat keras kuantum

Kita kemudian men-transpile Circuit menggunakan pass manager sesuai pola Qiskit standar.
Dengan menyediakan Backend yang mendukung fractional gates ke `generate_preset_pass_manager`, sebuah pass khusus bernama `FoldRzzAngle` dimasukkan secara otomatis.
Pass ini memodifikasi Circuit agar memenuhi batasan sudut RZZ.
Hasilnya, Gate RZZ dengan nilai negatif pada gambar sebelumnya diubah menjadi nilai positif, dan beberapa Gate X tambahan ditambahkan.

In [11]:
sampler_f = SamplerV2(mode=backend_f)
sampler_f.options.dynamical_decoupling.enable = True
sampler_f.options.dynamical_decoupling.sequence_type = "XY4"
sampler_f.options.dynamical_decoupling.skip_reset_qubits = True

In [12]:
job = sampler_f.run(t_qc_f, shots=shots)
print(job.job_id())

d4bninsi51bc738j97eg


### Step 4: Post-process and return result in desired classical format

You can obtain the kernel function value $K(x, x)$ by measuring the probability of the all-zero bitstring `00...00` in the output.

In [13]:
# job = service.job("d1obougt0npc73flhiag")
result = job.result()
fidelity_f = [fidelity(result=res) for res in result]
print(fidelity_f)
usage_f = job.usage()

[0.9005, 0.647, 0.3345, 0.355, 0.3315, 0.174, 0.1875, 0.149, 0.1175, 0.085]


![Output of the previous code cell](../docs/images/tutorials/fractional-gates/extracted-outputs/a18e5c70-1.avif)

Untuk menilai dampak fractional gates, kita mengevaluasi jumlah gate non-lokal (CZ dan RZZ untuk Backend ini),
beserta kedalaman dan durasi Circuit, dan membandingkan metrik-metrik ini dengan workflow standar nanti.

In [14]:
# step 1: map classical inputs to quantum problem
# `circuits` and `params` from the previous section are reused here

In [15]:
# step 2: optimize circuits
backend_c = service.backend(backend_name)  # w/o fractional gates
pm_c = generate_preset_pass_manager(
    optimization_level=optimization_level, backend=backend_c
)
t_qc_c = pm_c.run(circuits)
print(t_qc_c[0].count_ops())
t_qc_c[0].draw("mpl", fold=-1)

OrderedDict([('rz', 130), ('sx', 80), ('cz', 36), ('measure', 4), ('barrier', 2)])


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/a10f2d95-1.avif" alt="Output of the previous code cell" />

In [16]:
nnl_c = [qc.num_nonlocal_gates() for qc in t_qc_c]
depth_c = [qc.depth() for qc in t_qc_c]
duration_c = [
    qc.estimate_duration(backend_c.target, unit="u") for qc in t_qc_c
]

In [17]:
# step 3: execute
sampler_c = SamplerV2(backend_c)
sampler_c.options.dynamical_decoupling.enable = True
sampler_c.options.dynamical_decoupling.sequence_type = "XY4"
sampler_c.options.dynamical_decoupling.skip_reset_qubits = True

In [18]:
job = sampler_c.run(pubs=zip(t_qc_c, params), shots=shots)
print(job.job_id())

d4bnirvnmdfs73ae3a2g


In [19]:
# step 4: post-processing
# job = service.job("d1obp8j3rr0s73bg4810")
result = job.result()
fidelity_c = [fidelity(res) for res in result]
print(fidelity_c)
usage_c = job.usage()

[0.6675, 0.5725, 0.098, 0.102, 0.065, 0.0235, 0.006, 0.0015, 0.0015, 0.002]


## Perbandingan workflow dan Circuit tanpa fractional gates
Di bagian ini, kita menyajikan workflow Qiskit patterns standar menggunakan Backend yang tidak mendukung fractional gates.
Dengan membandingkan Circuit yang di-transpile, kamu akan melihat bahwa versi yang menggunakan fractional gates (dari bagian sebelumnya) lebih ringkas dibanding yang tanpa fractional gates.

In [20]:
plt.plot(qubits, depth_c, "-o", label="no fractional gates")
plt.plot(qubits, depth_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("depth")
plt.title("Comparison of depths")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/ef343a53-1.avif" alt="Output of the previous code cell" />

In [21]:
plt.plot(qubits, duration_c, "-o", label="no fractional gates")
plt.plot(qubits, duration_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("duration (µs)")
plt.title("Comparison of durations")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/98bb2cd0-1.avif" alt="Output of the previous code cell" />

In [22]:
plt.plot(qubits, nnl_c, "-o", label="no fractional gates")
plt.plot(qubits, nnl_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("number of non-local gates")
plt.title("Comparison of numbers of non-local gates")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/1383b242-1.avif" alt="Output of the previous code cell" />

In [23]:
plt.plot(qubits, fidelity_c, "-o", label="no fractional gates")
plt.plot(qubits, fidelity_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("fidelity")
plt.title("Comparison of fidelities")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/8b4594f5-1.avif" alt="Output of the previous code cell" />

We compare the QPU usage time with and without fractional gates. The results in the following cell show that the QPU usage times are almost identical.

In [24]:
print(f"no fractional gates: {usage_c} seconds")
print(f"fractional gates: {usage_f} seconds")

no fractional gates: 7 seconds
fractional gates: 7 seconds


## Advanced topic: Using only fractional RX gates

The need for the modified workflow when using fractional gates primarily stems from the restriction on RZZ gate angles.
However, if you use only the fractional RX gates and exclude the fractional RZZ gates, you can continue to follow the standard Qiskit patterns workflow.
This approach can still offer meaningful benefits, particularly in circuits that involve a large number of RX gates and U gates, by reducing the overall gate count and potentially improving performance.
In this section, we demonstrate how to optimize your circuits using only fractional RX gates, while omitting RZZ gates.

To support this, we provide a utility function that allows you to disable a specific basis gate in a Target object.
Here, we use it to disable RZZ gates.

In [25]:
from qiskit.circuit.library import n_local
from qiskit.transpiler import Target

In [26]:
def remove_instruction_from_target(target: Target, gate_name: str) -> Target:
    new_target = Target(
        description=target.description,
        num_qubits=target.num_qubits,
        dt=target.dt,
        granularity=target.granularity,
        min_length=target.min_length,
        pulse_alignment=target.pulse_alignment,
        acquire_alignment=target.acquire_alignment,
        qubit_properties=target.qubit_properties,
        concurrent_measurements=target.concurrent_measurements,
    )

    for name, qarg_map in target.items():
        if name == gate_name:
            continue
        instruction = target.operation_from_name(name)
        if qarg_map == {None: None}:
            qarg_map = None
        new_target.add_instruction(instruction, qarg_map, name=name)
    return new_target

We use a circuit consisting of U, CZ, and RZZ gates as an example.

In [27]:
qc = n_local(3, "u", "cz", "linear", reps=1)
qc.rzz(1.1, 0, 1)
qc.draw("mpl")

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/6b812497-0.avif" alt="Output of the previous code cell" />

We first transpile the circuit for a backend that does not support fractional gates.

In [28]:
pm_c = generate_preset_pass_manager(
    optimization_level=optimization_level, backend=backend_c
)
t_qc = pm_c.run(qc)
print(t_qc.count_ops())
t_qc.draw("mpl")

OrderedDict([('rz', 23), ('sx', 16), ('cz', 4)])


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/9e8e0709-1.avif" alt="Output of the previous code cell" />

## Perbandingan kedalaman dan fidelitas
Di bagian ini, kita membandingkan jumlah gate non-lokal dan fidelitas antara Circuit dengan dan tanpa fractional gates.
Ini menonjolkan potensi manfaat penggunaan fractional gates dari segi efisiensi eksekusi dan kualitas.

In [29]:
backend_f = service.backend(backend_name, use_fractional_gates=True)
target = remove_instruction_from_target(backend_f.target, "rzz")
pm_f = generate_preset_pass_manager(
    optimization_level=optimization_level,
    target=target,
)
t_qc = pm_f.run(qc)
print(t_qc.count_ops())
t_qc.draw("mpl")

OrderedDict([('rz', 22), ('sx', 14), ('cz', 4), ('rx', 1)])


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/db45feb0-1.avif" alt="Output of the previous code cell" />

### Optimize U gates with fractional RX gates

In this section, we demonstrate how to optimize U gates using fractional RX gates, building on the same circuit introduced in the previous section.

You will need to install the `qiskit-basis-constructor` [package](https://github.com/Qiskit/qiskit-basis-constructor) for this section.
This is a beta version of a new transpilation plugin for Qiskit, which might be integrated into Qiskit in the future.

In [30]:
# %pip install qiskit-basis-constructor

In [31]:
from qiskit.circuit.library import UGate
from qiskit_basis_constructor import DEFAULT_EQUIVALENCE_LIBRARY

We transpile the circuit using only fractional RX gates, excluding RZZ gates.
By introducing a custom decomposition rule, as shown in the following,
we can reduce the number of single-qubit gates required to implement a U gate.

This feature is currently under discussion in this [GitHub issue.](https://github.com/Qiskit/qiskit/issues/13455)

In [32]:
# special decomposition rule for UGate
x = ParameterVector("x", 3)
zxz = QuantumCircuit(1)
zxz.rz(x[2] - np.pi / 2, 0)
zxz.rx(x[0], 0)
zxz.rz(x[1] + np.pi / 2, 0)
DEFAULT_EQUIVALENCE_LIBRARY.add_equivalence(UGate(x[0], x[1], x[2]), zxz)

Next, we apply the transpiler using `constructor-beta` translation provided by `qiskit-basis-constructor` package.
As a result, the total number of gates is reduced compare to the previous transpilation.

In [33]:
pm_f = generate_preset_pass_manager(
    optimization_level=optimization_level,
    target=target,
    translation_method="constructor-beta",
)
t_qc = pm_f.run(qc)
print(t_qc.count_ops())
t_qc.draw("mpl")

OrderedDict([('rz', 16), ('rx', 9), ('cz', 4)])


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/b19aae7c-1.avif" alt="Output of the previous code cell" />

## Troubleshooting

### Issue: Invalid RZZ angles might remain after transpilation

As of Qiskit v2.0.3, there are known issues where RZZ gates with invalid angles may remain in the circuits even after transpilation.
The issue typically arises under the following conditions.

#### Failure when using `target` option with `generate_preset_pass_manager` or `transpiler`

When the `target` option is used with `generate_preset_pass_manager` or `transpiler`, the specialized transpiler pass `FoldRzzAngle` is not invoked.
To ensure proper handling of RZZ angles for fractional gates, we recommend always using the `backend` option instead.
See [this issue](https://github.com/Qiskit/qiskit/issues/14318) for more details.

#### Failure when circuits contain certain gates

If your circuit includes certain gates such as `XXPlusYYGate`, the Qiskit transpiler may generate RZZ gates with invalid angles.
If you encounter this issue, see this [GitHub issue](https://github.com/Qiskit/qiskit-ibm-runtime/issues/2256#issuecomment-2889487152) for a workaround.

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_cCNiGkGX5xZMzoG)